# Plan B - Datasource Report (Version 10)

This notebook uses **Power BI Admin APIs only** (no Scanner API) to produce
an end-to-end datasource inventory across the tenant, with Databricks-focused
classification and owner contact context.

## Version 10 architecture changes
- Replaced Scanner API datasource retrieval (`getInfo` / `scanStatus` / `scanResult`)
  with a direct admin pipeline:
  1. Enumerate workspaces via `GET /admin/groups`
  2. Enumerate datasets per workspace in parallel via `GET /admin/groups/{workspaceId}/datasets`
  3. Enumerate datasources per dataset in parallel via `GET /admin/datasets/{datasetId}/datasources`
- Added worker-local retry/backoff handling for throttling (429).
- Preserved output schema and Delta table compatibility:
  - `all_datasources`
  - `databricks_all_connections`
  - `databricks_personal_connections`


## Cell 1 — Setup and configuration

Initializes authentication, constants, and concurrency/retry settings.


In [ ]:
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests as http_requests

# Spark session is available automatically in Fabric notebooks
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

try:
    pbi_token = mssparkutils.credentials.getToken("https://analysis.windows.net/powerbi/api")
except Exception:
    pbi_token = mssparkutils.credentials.getToken("https://api.fabric.microsoft.com/")

HEADERS = {
    "Authorization": f"Bearer {pbi_token}",
    "Content-Type": "application/json",
}

PBI_API = "https://api.powerbi.com/v1.0/myorg"

# Key configuration
INCLUDE_PERSONAL_WORKSPACES = False
KNOWN_VNET_GATEWAY_IDS = {"3e481240-199c-402f-8fe3-ae5ef09b1ab4"}
MAX_WORKERS = 20
MAX_RETRIES = 5
THROTTLE_WAIT = 30
MAX_BACKOFF = THROTTLE_WAIT * 4

TABLE_ALL_DATASOURCES = "all_datasources"
TABLE_DATABRICKS_ALL = "databricks_all_connections"
TABLE_DATABRICKS_PERSONAL = "databricks_personal_connections"

print("Setup complete.")
print(f"Include personal workspaces: {INCLUDE_PERSONAL_WORKSPACES}")
print(f"Max workers: {MAX_WORKERS}, max retries: {MAX_RETRIES}, throttle wait: {THROTTLE_WAIT}s")


## Cell 2 — Load capacity metadata

Builds `cap_map` (`capacity_id -> capacity_name/capacity_sku`) for enrichment.


In [ ]:
print("Fetching capacities...")
cap_map = {}

try:
    cap_resp = http_requests.get(f"{PBI_API}/admin/capacities", headers=HEADERS)
    if cap_resp.status_code == 200:
        for cap in cap_resp.json().get("value", []):
            cap_map[cap["id"]] = {
                "capacity_name": cap.get("displayName", ""),
                "capacity_sku": cap.get("sku", ""),
            }
        print(f"{len(cap_map)} capacities loaded.")
    else:
        print(f"Status {cap_resp.status_code}: {cap_resp.text[:300]}")
except Exception as e:
    print(f"Error: {e}")


## Cell 3 — Step 1: Enumerate workspaces

Pages through `GET /admin/groups?$top=5000&$skip=N`, optionally filters
`PersonalGroup`, and builds `ws_map`.


In [ ]:
print("Fetching workspaces...")
raw_workspaces = []
top = 5000
skip = 0

while True:
    ws_resp = http_requests.get(
        f"{PBI_API}/admin/groups?$top={top}&$skip={skip}",
        headers=HEADERS,
    )

    if ws_resp.status_code == 429:
        retry = int(ws_resp.headers.get("Retry-After", THROTTLE_WAIT))
        retry = min(retry, MAX_BACKOFF)
        print(f"Rate limited while listing workspaces. Sleeping {retry}s...")
        time.sleep(retry)
        continue

    if ws_resp.status_code != 200:
        print(f"Status {ws_resp.status_code}: {ws_resp.text[:300]}")
        break

    batch = ws_resp.json().get("value", [])
    if not batch:
        break

    raw_workspaces.extend(batch)

    if len(batch) < top:
        break

    skip += top

print(f"Total workspaces fetched: {len(raw_workspaces)}")

if INCLUDE_PERSONAL_WORKSPACES:
    all_workspaces = raw_workspaces
    print("Including personal workspaces (flag is True).")
else:
    all_workspaces = [
        ws for ws in raw_workspaces
        if ws.get("type", "") != "PersonalGroup"
    ]
    excluded = len(raw_workspaces) - len(all_workspaces)
    print(f"Excluded {excluded} personal workspaces (type=PersonalGroup).")

print(f"Workspaces to process: {len(all_workspaces)}")

ws_map = {
    ws["id"]: {
        "workspace_name": ws.get("name", ""),
        "capacity_id": ws.get("capacityId", ""),
    }
    for ws in all_workspaces
}


## Cell 4 — Step 2: Enumerate datasets per workspace (parallel)

Calls `GET /admin/groups/{workspaceId}/datasets` with `ThreadPoolExecutor`.
Each worker handles retries/backoff for 429 and gracefully skips 404.


In [ ]:
print("Enumerating datasets per workspace in parallel...")


def _fetch_workspace_datasets(workspace_id, workspace_name, capacity_id):
    url = f"{PBI_API}/admin/groups/{workspace_id}/datasets"
    wait_seconds = THROTTLE_WAIT

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = http_requests.get(url, headers=HEADERS)

            if resp.status_code == 200:
                datasets = resp.json().get("value", [])
                out = []
                for ds in datasets:
                    out.append({
                        "workspace_id": workspace_id,
                        "workspace_name": workspace_name,
                        "capacity_id": capacity_id,
                        "dataset_id": ds.get("id", ""),
                        "dataset_name": ds.get("name", ""),
                        "configured_by": ds.get("configuredBy", ""),
                    })
                return out, "ok"

            if resp.status_code == 404:
                return [], "workspace_404"

            if resp.status_code == 429:
                retry_after = int(resp.headers.get("Retry-After", wait_seconds))
                sleep_for = min(retry_after, MAX_BACKOFF)
                time.sleep(sleep_for)
                wait_seconds = min(wait_seconds * 2, MAX_BACKOFF)
                continue

            return [], f"workspace_http_{resp.status_code}"

        except Exception:
            if attempt < MAX_RETRIES:
                time.sleep(min(wait_seconds, MAX_BACKOFF))
                wait_seconds = min(wait_seconds * 2, MAX_BACKOFF)
                continue
            return [], "workspace_exception"

    return [], "workspace_retry_exhausted"


all_datasets = []
workspace_warning_counts = {
    "workspace_404": 0,
    "workspace_other_errors": 0,
}

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {}
    for ws_id, meta in ws_map.items():
        futures[executor.submit(
            _fetch_workspace_datasets,
            ws_id,
            meta.get("workspace_name", ""),
            meta.get("capacity_id", ""),
        )] = ws_id

    total = len(futures)
    done = 0

    for future in as_completed(futures):
        done += 1
        rows, status = future.result()
        if rows:
            all_datasets.extend(rows)

        if status == "workspace_404":
            workspace_warning_counts["workspace_404"] += 1
        elif status != "ok":
            workspace_warning_counts["workspace_other_errors"] += 1

        if done % 100 == 0 or done == total:
            print(f"  Workspace progress: {done}/{total} processed; datasets collected: {len(all_datasets)}")

print(f"Total datasets enumerated: {len(all_datasets)}")
print(f"Workspace 404 skips: {workspace_warning_counts['workspace_404']}")
print(f"Workspace other-error skips: {workspace_warning_counts['workspace_other_errors']}")


## Cell 5 — Step 3: Fetch datasources per dataset (parallel)

Calls `GET /admin/datasets/{datasetId}/datasources` for every dataset with
per-worker retry/backoff. Empty datasource lists (push/streaming datasets) are skipped.


In [ ]:
print("Fetching dataset datasources in parallel...")


def _fetch_dataset_datasources(dataset):
    dataset_id = dataset["dataset_id"]
    url = f"{PBI_API}/admin/datasets/{dataset_id}/datasources"
    wait_seconds = THROTTLE_WAIT

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = http_requests.get(url, headers=HEADERS)

            if resp.status_code == 200:
                return resp.json().get("value", []), "ok"

            if resp.status_code == 404:
                return [], "dataset_404"

            if resp.status_code == 429:
                retry_after = int(resp.headers.get("Retry-After", wait_seconds))
                sleep_for = min(retry_after, MAX_BACKOFF)
                time.sleep(sleep_for)
                wait_seconds = min(wait_seconds * 2, MAX_BACKOFF)
                continue

            return [], f"dataset_http_{resp.status_code}"

        except Exception:
            if attempt < MAX_RETRIES:
                time.sleep(min(wait_seconds, MAX_BACKOFF))
                wait_seconds = min(wait_seconds * 2, MAX_BACKOFF)
                continue
            return [], "dataset_exception"

    return [], "dataset_retry_exhausted"


datasource_records = []
dataset_warning_counts = {
    "dataset_404": 0,
    "dataset_other_errors": 0,
    "dataset_no_datasources": 0,
}

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(_fetch_dataset_datasources, ds): ds for ds in all_datasets}

    total = len(futures)
    done = 0

    for future in as_completed(futures):
        done += 1
        dataset = futures[future]
        sources, status = future.result()

        if status == "dataset_404":
            dataset_warning_counts["dataset_404"] += 1
        elif status != "ok":
            dataset_warning_counts["dataset_other_errors"] += 1

        if status == "ok":
            if not sources:
                dataset_warning_counts["dataset_no_datasources"] += 1
            else:
                for src in sources:
                    datasource_records.append({
                        "dataset": dataset,
                        "datasource": src,
                    })

        if done % 100 == 0 or done == total:
            print(f"  Dataset progress: {done}/{total} processed; datasource rows collected: {len(datasource_records)}")

print(f"Datasets processed: {len(all_datasets)}")
print(f"Datasets with 404 skips: {dataset_warning_counts['dataset_404']}")
print(f"Datasets with other-error skips: {dataset_warning_counts['dataset_other_errors']}")
print(f"Datasets with zero datasources (skipped): {dataset_warning_counts['dataset_no_datasources']}")
print(f"Datasource objects collected: {len(datasource_records)}")


## Cell 6 — Step 4: Parse and classify datasource rows

Builds the final schema-compatible row set with Databricks classification,
gateway class, and connection parsing (`host`, `http_path`).


In [ ]:
print("Parsing and classifying datasources...")
results = []


def _normalize_connection_details(conn_raw):
    if isinstance(conn_raw, dict):
        return conn_raw
    if isinstance(conn_raw, str):
        txt = conn_raw.strip()
        if txt.startswith("{"):
            try:
                parsed = json.loads(txt)
                if isinstance(parsed, dict):
                    return parsed
            except Exception:
                pass
    return {}


for item in datasource_records:
    dataset = item["dataset"]
    src = item["datasource"]

    capacity_id = dataset.get("capacity_id", "")
    cap_info = cap_map.get(capacity_id, {})
    capacity_name = cap_info.get("capacity_name", "")
    capacity_sku = cap_info.get("capacity_sku", "")

    ds_type = src.get("datasourceType", "(unknown)")

    conn_raw = src.get("connectionDetails", {})
    conn_obj = _normalize_connection_details(conn_raw)

    if isinstance(conn_raw, dict):
        connection_details = json.dumps(conn_raw)
    elif isinstance(conn_raw, str):
        connection_details = conn_raw
    else:
        connection_details = json.dumps(conn_obj)

    gateway_id = src.get("gatewayId", "")

    ds_type_lower = str(ds_type).lower()
    connection_lc = connection_details.lower() if isinstance(connection_details, str) else ""
    is_databricks = (
        ds_type_lower in ("databricks", "azuredatabricks")
        or (ds_type_lower == "extension" and "databricks" in connection_lc)
    )

    if not gateway_id:
        gateway_class = "No Gateway"
    elif gateway_id in KNOWN_VNET_GATEWAY_IDS:
        gateway_class = "VNet Gateway"
    else:
        gateway_class = "Personal Cloud Gateway"

    host = conn_obj.get("host") or conn_obj.get("server") or conn_obj.get("path", "")
    http_path = conn_obj.get("httpPath") or conn_obj.get("database", "")

    if isinstance(host, str) and host.strip().startswith("{"):
        try:
            host_json = json.loads(host)
            if isinstance(host_json, dict):
                host = host_json.get("host") or host_json.get("server") or host
                if not http_path:
                    http_path = host_json.get("httpPath", "")
        except Exception:
            pass

    datasource_id = src.get("datasourceId") or src.get("id", "")

    results.append({
        "capacity_id": capacity_id,
        "capacity_name": capacity_name,
        "capacity_sku": capacity_sku,
        "workspace_id": dataset.get("workspace_id", ""),
        "workspace_name": dataset.get("workspace_name", ""),
        "dataset_id": dataset.get("dataset_id", ""),
        "dataset_name": dataset.get("dataset_name", ""),
        "datasource_id": datasource_id,
        "datasource_type": ds_type,
        "is_databricks": is_databricks,
        "gateway_id": gateway_id,
        "gateway_class": gateway_class,
        "host": host,
        "http_path": http_path,
        "connection_details": connection_details,
        "owner_name": dataset.get("configured_by", ""),
        "owner_email": dataset.get("configured_by", ""),
    })

print(f"Final result rows: {len(results)}")
print(f"Databricks rows: {sum(1 for r in results if r['is_databricks'])}")


## Cell 7 — Step 5: Write Delta tables and summary

Writes `all_datasources`, `databricks_all_connections`, and
`databricks_personal_connections` to the attached Lakehouse.


In [ ]:
df_all = pd.DataFrame(results)
df_dbx_all = pd.DataFrame()
df_dbx_personal = pd.DataFrame()

if df_all.empty:
    print("No datasources found. Nothing to write.")
else:
    # Convert is_databricks to string for Spark compatibility
    df_all["is_databricks"] = df_all["is_databricks"].map({True: "True", False: "False"})
    df_all = df_all.astype(str).fillna("")

    df_all = df_all.sort_values(
        by=["is_databricks", "gateway_class", "owner_email", "capacity_name", "workspace_name"],
        ascending=[False, False, True, True, True],
    ).reset_index(drop=True)

    df_dbx_all = df_all[df_all["is_databricks"] == "True"].copy()
    df_dbx_personal = df_dbx_all[df_dbx_all["gateway_class"] != "VNet Gateway"].copy()

    print("=" * 70)
    print("SUMMARY")
    print("=" * 70)

    print(f"\nAll datasources: {len(df_all)}")
    print(f"Databricks datasources: {len(df_dbx_all)}")
    print(f"Databricks not on VNet: {len(df_dbx_personal)}")

    print("\nDatasource types:")
    print(df_all.groupby("datasource_type").size().to_string())

    if not df_dbx_all.empty:
        print("\nDatabricks by gateway type:")
        print(df_dbx_all.groupby("gateway_class").size().to_string())

        print("\nDatabricks by capacity:")
        print(df_dbx_all.groupby(["capacity_name", "capacity_sku", "gateway_class"]).size().to_string())

    if not df_dbx_personal.empty:
        print("\nDatabricks non-VNet by owner:")
        print(df_dbx_personal.groupby(["owner_email", "gateway_class"]).size().to_string())
        print(f"\nUnique owners to contact: {df_dbx_personal['owner_email'].nunique()}")

    spark_df = spark.createDataFrame(df_all)
    (
        spark_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(TABLE_ALL_DATASOURCES)
    )
    print(f"\nWritten {len(df_all)} rows -> '{TABLE_ALL_DATASOURCES}'")

    if not df_dbx_all.empty:
        spark_dbx = spark.createDataFrame(df_dbx_all)
        (
            spark_dbx.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(TABLE_DATABRICKS_ALL)
        )
        print(f"Written {len(df_dbx_all)} rows -> '{TABLE_DATABRICKS_ALL}'")
    else:
        print(f"No Databricks datasources - '{TABLE_DATABRICKS_ALL}' not updated.")

    if not df_dbx_personal.empty:
        spark_pers = spark.createDataFrame(df_dbx_personal)
        (
            spark_pers.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(TABLE_DATABRICKS_PERSONAL)
        )
        print(f"Written {len(df_dbx_personal)} rows -> '{TABLE_DATABRICKS_PERSONAL}'")
    else:
        print(f"No personal/non-VNet Databricks connections - '{TABLE_DATABRICKS_PERSONAL}' not updated.")

    print("\nDone. Tables are now queryable via the Lakehouse SQL endpoint.")


## Cell 8 — Step 6: Optional email templates

Generates owner-targeted email text for Databricks datasets not on VNet Gateway.


In [ ]:
# Optional: print email templates for Databricks dataset owners
if not df_all.empty and not df_dbx_personal.empty:
    print("=" * 70)
    print("EMAIL TEMPLATES")
    print("=" * 70)

    for owner_email, grp in df_dbx_personal.groupby("owner_email"):
        datasets_list = "\n".join([
            f"   - {row['dataset_name']}\n"
            f"          (workspace: {row['workspace_name']}, capacity: {row['capacity_name']})"
            for _, row in grp.iterrows()
        ])
        hosts = grp["host"].unique()
        paths = grp["http_path"].unique()

        print(f"\n{'─' * 60}")
        print(f"To: {owner_email}")
        print("Subject: Action Required - Migrate Databricks connection")
        print("")
        print(f"Hi {grp.iloc[0]['owner_name']},")
        print("")
        print("These datasets need migration to VNet Data Gateway:")
        print("")
        print(datasets_list)
        print("")
        print(f"Host(s): {', '.join(str(h) for h in hosts)}")
        print(f"Path(s): {', '.join(str(p) for p in paths)}")
        print("")
        print("Steps:")
        print("  1. Power BI > Settings > Manage connections and gateways")
        print("  2. Select VNet Data Gateway")
        print("  3. Add data source > Databricks > enter host + HTTP path")
        print("  4. Sign in with OAuth2")
        print("  5. Dataset settings > Gateway > select VNet datasource")
        print("  6. Refresh dataset")
        print(f"{'─' * 60}")
else:
    print("No personal Databricks connections to report.")
